In [1]:
import pandas as pd
import numpy as np

In [2]:
# Perform uniform sampling with replacement from a pool of 0s and 1s
def uniform_sampling(row):
    ref_depth = row['orig_DEPTH'] - row['ALT_DEPTH']
    pool = [1] * row['ALT_DEPTH'] + [0] * ref_depth
    sampled = np.random.choice(pool, size=row['DEPTH'], replace=False)
    return sampled.sum()

def downsample_mutations(table, proportion, samplename):
    """
    Downsample mutation depths and filter mutations based on VAF.

    FILE format: tab-delimited with columns:
    """

    if not (0 < proportion <= 1):
        click.echo("Error: Proportion must be between 0 and 1.")
        return

    df = table.rename(columns={"DEPTH": "orig_DEPTH"})

    # Downsample the depth values by multiplying and rounding
    df["DEPTH"] = ((df["orig_DEPTH"] * proportion) // 1).astype(int)

    # Use Poisson distribution to decide if mutation remains
    #df['upd_ALT_DEPTH'] = df.apply(lambda row: np.random.poisson(row['VAF'], row['DEPTH']).sum(), axis=1)
    df['ALT_DEPTH'] = df.apply(uniform_sampling, axis=1)
    df['retain_mutation'] = df['ALT_DEPTH'] > 0

    # Filter to retain only mutations
    filtered_df = df[df['retain_mutation']].drop(columns=['retain_mutation', 'orig_DEPTH'])
    filtered_df["VAF"] = df['ALT_DEPTH'] / df["DEPTH"]

    return filtered_df


In [ ]:
# annotatedepths/all_samples_indv.depths.tsv.gz
all_depths = pd.read_csv("all_samples_indv.depths.tsv.gz", sep="\t")

# pipeline_info/samplesheet.valid.csv
input_samples = pd.read_csv("samplesheet.valid.csv", sep=",")

In [4]:
all_depths.iloc[:,3:].sum().sum()

np.int64(106765162290)

In [5]:
# downsample the following samples to 65% I want to keep all the samples in the same final table
samples_to_downsample = ['L844', 'L845', 'L846', 'L847', 'L848', 'L849', 'L850', 'L851', 'L855', 'L857', 'L858', 'L860', 'L861', 'L862', 'L863', 'L864', 'L865', 'L866', 'L867', 'L868', 'L869', 'L870', 'L871', 'L872', 'L874', 'L875', 'L876', 'L877', 'L878', 'L879', 'L880', 'L881', 'L882', 'L883', 'L884', 'L885', 'L886', 'L887', 'L888', 'L889', 'L890', 'L891', 'L893', 'L903', 'L925', 'L931', 'L937']
proportion = 0.65
for sample in samples_to_downsample:
    # Downsample the depth values by multiplying and rounding
    all_depths[sample] = ((all_depths[sample] * proportion) // 1).astype(int)


In [6]:
all_depths.iloc[:,3:].sum().sum()

np.int64(98896928449)

In [7]:
# downsample the following samples to 65% I want to keep all the samples in the same final table
samples_to_downsample = ['L136', 'L138', 'L140', 'L143', 'L146', 'L148', 'L150', 'L152', 'L184', 'L187', 'L188', 'L326', 'L328', 'L330', 'L560', 'L562', 'L563']
proportion = 0.4
for sample in samples_to_downsample:
    # Downsample the depth values by multiplying and rounding
    all_depths[sample] = ((all_depths[sample] * proportion) // 1).astype(int)


In [8]:
all_depths.iloc[:,3:].sum().sum()

np.int64(93080282078)

In [9]:
input_samples['bam_name'] = input_samples['bam'].apply(lambda x: x.split('/')[-1])
input_sample_ids_bam = dict(input_samples[['sample', 'bam_name']].values)
input_sample_ids_bam

{'L128': '128.sorted.bam',
 'L130': '130.sorted.bam',
 'L132': '132.sorted.bam',
 'L133': '133.sorted.bam',
 'L134': '134.sorted.bam',
 'L136': '136.sorted.bam',
 'L138': '138.sorted.bam',
 'L140': '140.sorted.bam',
 'L143': '143.sorted.bam',
 'L144': '144.sorted.bam',
 'L146': '146.sorted.bam',
 'L148': '148.sorted.bam',
 'L150': '150.sorted.bam',
 'L152': '152.sorted.bam',
 'L156': '156.sorted.bam',
 'L157': '157.sorted.bam',
 'L158': '158.sorted.bam',
 'L184': '184.sorted.bam',
 'L186': '186.sorted.bam',
 'L187': '187.sorted.bam',
 'L188': '188.sorted.bam',
 'L324': '324.sorted.bam',
 'L326': '326.sorted.bam',
 'L328': '328.sorted.bam',
 'L330': '330.sorted.bam',
 'L337': '337.sorted.bam',
 'L341': '341.sorted.bam',
 'L345': '345.sorted.bam',
 'L348': '348.sorted.bam',
 'L350': '350.sorted.bam',
 'L351': '351.sorted.bam',
 'L352': '352.sorted.bam',
 'L353': '353.sorted.bam',
 'L354': '354.sorted.bam',
 'L355': '355.sorted.bam',
 'L356': '356.sorted.bam',
 'L357': '357.sorted.bam',
 

In [10]:
all_depths_updated = all_depths.rename(input_sample_ids_bam, axis=1).drop("CONTEXT", axis=1)

In [11]:
all_depths_updated.head()

,CHROM,POS,128.sorted.bam,130.sorted.bam,132.sorted.bam,133.sorted.bam,134.sorted.bam,136.sorted.bam,138.sorted.bam,140.sorted.bam,...,930.sorted.bam,931.sorted.bam,932.sorted.bam,933.sorted.bam,934.sorted.bam,935.sorted.bam,936.sorted.bam,937.sorted.bam,938.sorted.bam,939.sorted.bam
0,chr1,26760334,0,1,2,0,1,0,0,0,...,117,102,115,97,0,69,116,63,0,142
1,chr1,26760335,0,1,2,0,1,0,0,0,...,117,102,115,97,0,69,117,63,0,142
2,chr1,26760336,1,1,2,0,1,0,0,0,...,117,103,115,99,0,70,116,63,0,142
3,chr1,26760337,1,1,2,0,1,0,0,0,...,117,102,116,99,0,70,116,63,0,142
4,chr1,26760338,1,1,2,0,1,0,0,0,...,117,103,116,100,0,70,116,63,0,142


In [12]:
all_depths_updated.to_csv("all_samples_indv.depths.downsampled.tsv.gz", sep="\t", index=False)